# Ejercicio

- ¿Puedes identificar los patrones de diseño de Agentic que se han usado aquí?
- ¿Cuál es la línea que cambió esto de ser un "flujo de trabajo" de Agentic a un "agente" según la definición de Anthropic?
- ¡Intenta agregar más herramientas y agentes! Podrías tener herramientas que gestionen la combinación de correspondencia para enviar a una lista.

**RETO DIFÍCIL:**
Investiga cómo puedes hacer que SendGrid llame a un webhook de devolución de llamada cuando un usuario responde a un correo electrónico. Luego, haz que el SDR responda para continuar la conversación. Esto puede requerir algo de programación de ambiente. 😅

---

# Implicaciones comerciales

Esto se aplica inmediatamente a la automatización de ventas; pero, de forma más general, podría aplicarse a la automatización integral de cualquier proceso empresarial mediante conversaciones y herramientas.
Piensa en cómo podrías aplicar una solución de agente como esta en tu trabajo diario.

### --Client Config

In [8]:
from agents import Runner, trace, Agent, function_tool, AsyncOpenAI, OpenAIChatCompletionsModel
from sendgrid.helpers.mail import Mail, To, Content, Email
from dotenv import load_dotenv
from typing import Dict
import sendgrid
import os

In [9]:
load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
LOCAL_LLAMA_BASE_URL = "http://localhost:11434/v1"

google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

if google_api_key:
    print(f'Google API Key: {google_api_key[:8]}')
else:
    print('Google API Key not set')
if groq_api_key:
    print(f'GroQ API Key: {groq_api_key[:8]}')
else:
    print('GroQ API Key not set')

Google API Key: AIzaSyCl
GroQ API Key: gsk_oFCs


### --Models / Agents

In [10]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

gemini_model = OpenAIChatCompletionsModel(model="gemini-2.0-flash", openai_client=gemini_client)

In [11]:
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

groq_model = OpenAIChatCompletionsModel(model="llama-3.1-8b-instant", openai_client=groq_client)

In [12]:
local_llama_client = AsyncOpenAI(base_url=LOCAL_LLAMA_BASE_URL, api_key="ollama")

local_llama_model = OpenAIChatCompletionsModel(model="llama3.1:8b", openai_client=local_llama_client)

In [13]:
instructions1 = "Eres un agente de ventas que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos profesionales y serios."

instructions2 = "Eres un agente de ventas atractivo y entretenido que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos ingeniosos y atractivos que probablemente obtengan respuesta."

instructions3 = "Eres un agente de ventas muy activo que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos concisos y directos."

In [14]:
sales_agent1 = Agent(name="Agente de ventas Groq", instructions=instructions1, model=groq_model)
sales_agent2 = Agent(name="agente de ventas Gemini", instructions=instructions2, model=gemini_model)
sales_agent3 = Agent(name="Agente de ventas Local Llama3.1", instructions=instructions3, model=local_llama_model)

In [15]:
subject_instructions = "Puedes escribir un asunto para un correo electronico de ventas en frio.\
Se te proporciona un mensaje y necesitas escribir un asunto para un correo electronico que probablemente obtenga respuesta"

html_istructions = "Puedes convertir un cuerpo de correo electronico de texto a un cuerpo de correo electronico HTML. \
    Se te proporciona un cuerpo de correo electronico de texto que puede tener algun markdown y necesitas convertirlo a un cuerpo\
         de correo electronico HTML con un diseño claro, atractivo y que se adecuado para el asunto."

In [16]:
subject_writer = Agent(name="Escritor de asunto de correo electronico",
                       instructions=subject_instructions, model=gemini_model)

html_converter = Agent(name="Conversor de cuerpo de correo electronico HTML",
                       instructions=html_istructions, model=gemini_model)

### --Tools & Handoffs

In [17]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envía un correo electrónico con el asunto y el cuerpo HTML a todos los cleintes potenciales de ventas."""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv('SENDGRID_API_KEY'))
    from_email = Email("marco.salram.04@gmail.com")  # Cambiar a tu remitente verificado
    to_email = To("adr.slm.rmz@gmail.com")  # Cambiar a su receptor
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    response = sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [18]:
description = "Escribe un correo electronico de ventas en frio"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [19]:
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Escribe un asunto para un correo electronico de ventas en frio")

html_tool = html_converter.as_tool(tool_name="html_converter",
                                   tool_description="Convierte un cuerpo de correo electronico de texto a un cuerpo de correo electronico HTML")

In [20]:
email_tools = [subject_tool, html_tool, send_html_email]

In [21]:
tools = [tool1, tool2, tool3]

In [22]:
instructions = "Eres un formateador y remitente de correos electronicos.\
    Recibes el cuerpo de un correo electronico para enviarlo.\
    Primero usas la herrameinta subject_writer para escribir un asunto para el correo electronico,\
    luego usas la herramienta html_converter para convertir el cuerpo a HTML.\
    Finalmente, usas la herramienta send_html_email para enviar el correo electronico con el asunto y el cuerpo HTML."

emailer_agent = Agent(
    name = "Email Manager",
    instructions=instructions,
    tools=email_tools,
    model = gemini_model,
    handoff_description="Convierte un email a HTML y lo envia"
)

In [23]:
handoffs = [emailer_agent]

sales_manager = Agent(
    name = "Manager de ventas",
    tools=tools,
    handoffs=handoffs,
    model=gemini_model
)

### --Input Guardrails

In [24]:
from pydantic import BaseModel, Field, field_validator
from agents import WebSearchTool, input_guardrail, GuardrailFunctionOutput
import re

In [26]:
class EmailCheckOutput(BaseModel):
    subject: str = Field(description="Validar asunto del email", min_length=3, max_length=100)
    body: str = Field(description="Body del email", min_length=10)
    to: str = Field(description="Email destinatario")

    @field_validator("body")
    def check_for_secrets(cls, v):
        # Patrón simple para detectar posibles API keys o contraseñas
        secret_patterns = [
            r"api[_-]?key\s*=\s*\w+",
            r"secret\s*=\s*\w+",
            r"password\s*=\s*\w+",
            r"Bearer\s+[A-Za-z0-9\-_\.]+"
        ]
        for pattern in secret_patterns:
            if re.search(pattern, v, re.IGNORECASE):
                raise ValueError("El cuerpo contiene posibles credenciales o API keys.")
        return v

guardrail_agent = Agent(
    name="Revision de asunto y claves",
    instructions="Revisa si el usuario esta incluyendo claves sensibles al mandar un asunto mediante correo electronico",
    output_type=EmailCheckOutput,
    model=gemini_model
)

In [27]:
@input_guardrail
async def guardrail_against_keys(ctx, agent, message):
    try:
        result = await Runner.run(guardrail_agent, message, context=ctx.context)
        return GuardrailFunctionOutput(
            output_info={"validated_email": result.final_output},
            tripwire_triggered=False  # no se encontraron claves
        )
    except ValueError as e:
        return GuardrailFunctionOutput(
            output_info={"error": str(e)},
            tripwire_triggered=True   # se detectaron claves/API keys
        )

In [30]:
sales_manager_instructions = "Eres un gerente de ventas que trabaja para ComplAI. Utilizas las herramientas que se te proporcionan para generar correos electrónicos de ventas en frío. \
Nunca generas correos electrónicos de ventas tú mismo; siempre utilizas las herramientas. \
Pruebas las 3 herramientas del agente de ventas al menos una vez antes de elegir la mejor. \
Puedes usar las herramientas múltiples veces si no estás satisfecho con los resultados del primer intento. \
Seleccionas el mejor correo electrónico usando tu propio criterio sobre cuál será más efectivo. \
Después de elegir el correo electrónico, transfieres al agente Email Manager para formatear y enviar el correo."

careful_sales_manager = Agent(
    name="Manager de ventas",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=[emailer_agent],
    model=gemini_model,
    input_guardrails=[guardrail_against_keys]
)

### --Pruebas

In [31]:
message = "Envia un correo de ventas en frio dirigido a 'Estimado director ejecutivo de Ciime' desde Marco"

with trace("Protected Automated SDR"):
    result = await Runner.run(careful_sales_manager, message)

RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 32.893685611s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goog

OPENAI_API_KEY is not set, skipping trace export
